# Project 1 — Healthcare Readmission Risk Analytics
## Notebook 1: Exploratory Data Analysis (EDA)

**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008)  
**Rows:** ~101,766 hospital admissions  
**Goal:** Understand the data structure, distributions, and missing values before modelling

**Skills demonstrated:** pandas, matplotlib, seaborn, data profiling, missing value strategy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/diabetic_readmission.csv', low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

## 2. Basic Profile

In [ ]:
print('=== Data Types ===')
print(df.dtypes.value_counts())
print('\n=== Numeric Summary ===')
df.describe().round(2)

## 3. Missing Value Analysis

In [ ]:
# Replace '?' sentinel values with NaN
df.replace('?', np.nan, inplace=True)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df.missing_count > 0].sort_values('missing_pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(missing_df.index, missing_df.missing_pct, color='#E07B54')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
for bar, val in zip(bars, missing_df.missing_pct):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../powerbi/screenshots/missing_values.png', bbox_inches='tight')
plt.show()
print(missing_df)

## 4. Target Variable Distribution

In [ ]:
readmit_counts = df['readmitted'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c', '#3498db']
axes[0].bar(readmit_counts.index, readmit_counts.values, color=colors)
axes[0].set_title('Readmission Counts', fontweight='bold')
axes[0].set_xlabel('Readmission Status')
axes[0].set_ylabel('Number of Admissions')
for i, v in enumerate(readmit_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(readmit_counts.values, labels=readmit_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Readmission Distribution', fontweight='bold')

plt.suptitle('Target Variable: Hospital Readmission Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../powerbi/screenshots/readmission_distribution.png', bbox_inches='tight')
plt.show()

print(f'\n30-day readmission rate: {readmit_counts["<30"] / len(df) * 100:.1f}%')

## 5. Age Group Distribution vs Readmission

In [ ]:
age_readmit = df.groupby('age')['readmitted'].apply(
    lambda x: (x == '<30').sum() / len(x) * 100
).reset_index()
age_readmit.columns = ['age_group', 'readmit_rate_pct']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(age_readmit.age_group, age_readmit.readmit_rate_pct,
        marker='o', color='#2980b9', linewidth=2.5, markersize=8)
ax.fill_between(age_readmit.age_group, age_readmit.readmit_rate_pct,
                alpha=0.15, color='#2980b9')
ax.set_title('30-Day Readmission Rate by Age Group', fontsize=13, fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Readmission Rate (%)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../powerbi/screenshots/readmit_by_age.png', bbox_inches='tight')
plt.show()

## 6. Numeric Feature Distributions

In [ ]:
numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                'num_medications', 'number_outpatient', 'number_inpatient', 'number_emergency']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='#5DADE2', edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=9)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

axes[-1].axis('off')
plt.suptitle('Numeric Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../powerbi/screenshots/numeric_distributions.png', bbox_inches='tight')
plt.show()

## 7. Load into SQLite and Populate Star Schema

In [ ]:
conn = sqlite3.connect('../data/healthcare.db')

# Load schema
with open('../sql/01_schema.sql') as f:
    conn.executescript(f.read())

# Load raw CSV into staging table
df.to_sql('staging_raw', conn, if_exists='replace', index=False)

# Run data load script
with open('../sql/02_data_load.sql') as f:
    conn.executescript(f.read())

# Create views
with open('../sql/04_views.sql') as f:
    conn.executescript(f.read())

conn.commit()

# Verify row counts
for table in ['dim_patient', 'dim_diagnosis', 'fact_admissions']:
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count:,} rows')

conn.close()
print('\nSQLite database ready at ../data/healthcare.db')

## Key EDA Findings

| Finding | Value |
|---|---|
| Total admissions | ~101,766 |
| 30-day readmission rate | ~11% |
| Most common primary diagnosis | Circulatory (~30%) |
| Highest missing data column | `weight` (~97% missing) |
| Age group with highest readmission | `[70-80)` |

**Next:** Notebook 2 — Statistical Analysis to validate these patterns